# 02 · Tahmin modelleri ve FLAML

LightGBM, Random Forest ve CatBoost aynı tarih ayrımıyla karşılaştırılır. Bu modeller küme
oluşturmaz; üretim tahmini ve açıklayıcı değişken değerlendirmesi içindir. FLAML denemelerinden
**üç göstergenin banka geneli doğrulama WAPE ortalamasına göre tek algoritma ailesi** seçilir.
Gösterge / grup bazında eğitimler ayrı olabilir, algoritma ailesi ortak kalır.


In [ ]:
from pathlib import Path
import sys
_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "python_kodlari/ayarlar").is_dir() and (p / "notebooks/jupyter_araclari.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Projenin tamamını indirin; notebook’u projenin notebooks klasöründen açın.")
for _folder in [ROOT / "python_kodlari", ROOT / "notebooks"]:
    if str(_folder) not in sys.path:
        sys.path.insert(0, str(_folder))
from IPython.display import display, Markdown
from jupyter_araclari import (read_json, read_table, latest_run, fresh_run, record_run,
    result_path, require_files, prepare_configs, source_input, run_python, extract_bundle,
    snapshot, current_decision, qualified_selection)
print("Proje:", ROOT)
print("Kernel Python:", sys.executable)


In [ ]:
import pandas as pd
MODELLERI_EGIT = False
FLAML_ARAMASI_YAP = False
ORTAK_MODELI_EGIT = False
ARSIVDEN_MODEL_KAYNAGI_AC = False
ARAMA_BUTCESI_SANIYE = 120.0  # Her gösterge × grup araması için
HAM_VERI_KLASORU = ROOT / "outputs/banka_test_verisi_v4"
MODEL_KAYNAGI = latest_run("model_karsilastirma_v4_veri") or latest_run("model_karsilastirma_v4")
FLAML_ARAMA_KAYNAGI = latest_run("flaml_v4")
if ARSIVDEN_MODEL_KAYNAGI_AC:
    MODEL_KAYNAGI = extract_bundle("model_karsilastirma_v4_paketi.zip")
print("Model girdisi:", MODEL_KAYNAGI)


## İsteğe bağlı: aynı veriyle üç algoritmayı eğit

Önce 01’de eğitim setini hazırla. Arşiv seçeneği daha önce eğitilmiş model paketini yeni bir
klasöre açar; hazır sonuçları incelemek için gerekli değildir. Alt süreçte kernel’in Python’u
kullanılır; V4 protokolünün global ayarları notebook kernel’ine taşınmaz.


In [ ]:
if MODELLERI_EGIT:
    if MODEL_KAYNAGI is None:
        raise FileNotFoundError("Önce 01’de eğitim setini hazırlayın veya model arşivini açın.")
    require_files(*(MODEL_KAYNAGI / "veri" / f"{n}.csv" for n in ["train", "validation", "test", "ocak2027_tahmin_girdisi"]))
    require_files(*(HAM_VERI_KLASORU / n for n in ["manifest.json", "sube_model_girdileri_aylik.csv", "sube_stok_mutabakat.csv", "sube_aylik.csv"]))
    trained = fresh_run("model_karsilastirma_v4")
    run_python("model_karsilastir.py", "--protocol", "v4", "--source", MODEL_KAYNAGI, "--output", trained)
    run_python("model_karsilastir_dogrula.py", "--source", MODEL_KAYNAGI, "--output", trained, "--raw-source", HAM_VERI_KLASORU)
    record_run("model_karsilastirma_v4", trained)
    MODEL_KAYNAGI = trained
else:
    print("Üç algoritma yeniden eğitilmedi; hazır sonuçlar gösteriliyor.")


In [ ]:
metrics = read_table(result_path("model_karsilastirma_v4", "sonuclar/toplam_metrikler.csv", "v4_toplam_metrikler.csv"))
display(metrics[["gosterge", "model", "split", "wape", "mae_tl"]].sort_values(["split", "gosterge", "wape"]).head(24))
valid = metrics[metrics["split"].eq("VALIDATION")]
display(valid.pivot_table(index="model", columns="gosterge", values="wape"))


## FLAML araması ve ortak algoritma seçimi

Önce doğrulanmış üç algoritma eğitim çıktısı gerekir. Arama her hücre için adayları kaydeder;
bu ara deneyin hücre bazlı kazananı nihai politika değildir. Sonraki hücre bütün göstergeler
için tek algoritmayı seçer. Test sonuçları seçim puanına girmez. İncelenmiş tarihsel testin
sonucuna bakıp algoritma veya eşik değiştirmek yeni test başarısı sayılmaz.


In [ ]:
if FLAML_ARAMASI_YAP:
    if MODEL_KAYNAGI is None:
        raise FileNotFoundError("Önce doğrulanmış üç algoritma model çıktısını oluşturun.")
    require_files(MODEL_KAYNAGI / "manifest.json")
    searched = fresh_run("flaml_v4")
    run_python("flaml_pipeline.py", "--source", MODEL_KAYNAGI, "--output", searched, "--budget", ARAMA_BUTCESI_SANIYE)
    run_python("flaml_dogrula.py", "--source", MODEL_KAYNAGI, "--output", searched)
    record_run("flaml_v4", searched)
    FLAML_ARAMA_KAYNAGI = searched
else:
    print("Yeni FLAML araması başlatılmadı.")


In [ ]:
if ORTAK_MODELI_EGIT:
    if MODEL_KAYNAGI is None or FLAML_ARAMA_KAYNAGI is None:
        raise FileNotFoundError("Doğrulanmış model ve FLAML arama çıktıları gerekli.")
    from flaml_ortak_model import run as train_common
    from flaml_ortak_dogrula import main as verify_common
    common = fresh_run("flaml_ortak_v4")
    train_common(MODEL_KAYNAGI, FLAML_ARAMA_KAYNAGI, common)
    verify_common(common, MODEL_KAYNAGI, FLAML_ARAMA_KAYNAGI)
    record_run("flaml_ortak_v4", common)
else:
    print("Ortak model yeniden eğitilmedi.")


In [ ]:
common_run = latest_run("flaml_ortak_v4")
global_decision = read_json(common_run / "ortak_model_secimi.json") if common_run else snapshot()["automl"]["global_selection"]
display(pd.DataFrame(global_decision["algorithm_scores"]))
print("Bütün göstergelerde ortak algoritma:", global_decision["selected_learner"])
print("Seçim yalnızca doğrulamada:", global_decision["selection_uses_validation_only"])
assert global_decision["selection_policy"] == "one_algorithm_all_indicators_and_groups"


## Kaydedilmiş modelden tahmin

Metadata dosyası ile **aynı eğitim sözlüğüyle hazırlanmış** bir CSV seç. Müşteri ham verisi
doğrudan tahmin girdisi değildir. Bu hücre tekrar eğitim yapmaz; Jupyter’da tahminleri inceler.


In [ ]:
TAHMIN_URET = False
MODEL_METADATA = None  # Örnek: common_run / "modeller/tl_vadesiz__UC_BUYUK_IL__metadata.json"
TAHMIN_GIRDISI = None  # Örnek: common_run / "veri/ocak2027_tahmin_girdisi.csv"
if TAHMIN_URET:
    if MODEL_METADATA is None or TAHMIN_GIRDISI is None:
        raise ValueError("Metadata ve hazırlanmış tahmin CSV yolunu girin.")
    from flaml_tahmin import predict_saved
    frame = read_table(TAHMIN_GIRDISI)
    metadata = read_json(MODEL_METADATA)
    model_path = Path(MODEL_METADATA).parent / metadata["production_file"]
    predictions = predict_saved(model_path, frame)
    display(predictions.head())
